In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             auc, precision_score, recall_score, f1_score, accuracy_score)

from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup)

# Mount Google Drive (if using Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# Read the data files
df_attack_free = pd.read_csv('/content/drive/MyDrive/CAN_Research/CANData/CAND/Impala/attack-free-1.csv')
df_standstill = pd.read_csv('/content/drive/MyDrive/CAN_Research/CANData/CAND/Impala/standstill-1.csv')

# Process the attack-free dataset
df_attack_free = df_attack_free.loc[:50000].copy()  # First 50,001 rows
df_attack_free['datetime'] = df_attack_free['timestamp']
df_attack_free['arbitration_id'] = df_attack_free['arbitration_id'].astype(str)
df_attack_free['data_field'] = df_attack_free['data_field'].astype(str)
df_attack_free['attack'] = 0
df_attack_free = df_attack_free[['datetime', 'arbitration_id', 'data_field', 'attack']]

# Process the standstill dataset
df_standstill = df_standstill.loc[:50000].copy()
df_standstill['datetime'] = df_standstill['timestamp']
df_standstill['arbitration_id'] = df_standstill['arbitration_id'].astype(str)
df_standstill['data_field'] = df_standstill['data_field'].astype(str)
df_standstill['attack'] = 1
df_standstill = df_standstill[['datetime', 'arbitration_id', 'data_field', 'attack']]

# Combine and shuffle the data
data = pd.concat([df_attack_free, df_standstill], ignore_index=True)
data = shuffle(data, random_state=42).reset_index(drop=True)

# Prepare the feature string by combining the relevant columns
data_columns = ['datetime', 'arbitration_id', 'data_field']
data['feature_string'] = data[data_columns].apply(lambda x: '/'.join(x.astype(str)), axis=1)

# Extract features and labels
sentences = data['feature_string'].values
labels = data['attack'].values

# Set device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

# Tokenize the sentences using the tokenizer
max_length = 128  # Adjusted sequence length for better performance
encoding = tokenizer(
    sentences.tolist(),
    add_special_tokens=True,
    max_length=max_length,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

input_ids = encoding['input_ids']
attention_masks = encoding['attention_mask']
labels = torch.tensor(labels)

# Split the data into training, validation, and test sets
train_inputs, test_inputs, train_masks, test_masks, train_labels, test_labels = train_test_split(
    input_ids, attention_masks, labels,
    random_state=42, test_size=0.2
)

train_inputs, val_inputs, train_masks, val_masks, train_labels, val_labels = train_test_split(
    train_inputs, train_masks, train_labels,
    random_state=42, test_size=0.25
)

# Create DataLoaders
batch_size_train = 32
batch_size_val = 64

train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
val_dataset = TensorDataset(val_inputs, val_masks, val_labels)
test_dataset = TensorDataset(test_inputs, test_masks, test_labels)

train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size_train
)

validation_dataloader = DataLoader(
    val_dataset,
    sampler=SequentialSampler(val_dataset),
    batch_size=batch_size_val
)

test_dataloader = DataLoader(
    test_dataset,
    sampler=SequentialSampler(test_dataset),
    batch_size=batch_size_val
)

# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2
)
model.to(device)

# Set up the optimizer and learning rate scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs = 10
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Early stopping criteria
patience = 2
early_stopping_counter = 0
best_validation_loss = float('inf')

# Training loop
loss_values = []
validation_loss_values = []
training_start_time = time.time()

for epoch_i in range(epochs):
    print(f'======== Epoch {epoch_i + 1} / {epochs} ========')
    print('Training...')

    model.train()
    total_loss = 0

    progress_bar = tqdm(train_dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)

        model.zero_grad()

        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({'loss': loss.item()})

    avg_train_loss = total_loss / len(train_dataloader)
    loss_values.append(avg_train_loss)

    print(f"Average training loss: {avg_train_loss:.4f}")

    # Validation
    print("Running Validation...")

    model.eval()
    eval_loss = 0
    eval_accuracy = 0
    nb_eval_steps = 0

    for batch in validation_dataloader:
        b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)

        with torch.no_grad():
            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

        loss = outputs.loss
        logits = outputs.logits

        eval_loss += loss.item()

        preds = torch.argmax(logits, dim=1).flatten()
        labels = b_labels.flatten()

        accuracy = (preds == labels).cpu().numpy().mean() * 100
        eval_accuracy += accuracy

        nb_eval_steps += 1

    avg_val_loss = eval_loss / nb_eval_steps
    avg_val_accuracy = eval_accuracy / nb_eval_steps
    validation_loss_values.append(avg_val_loss)

    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {avg_val_accuracy:.2f}%")

    # Early stopping
    if avg_val_loss < best_validation_loss:
        best_validation_loss = avg_val_loss
        early_stopping_counter = 0
        # Save the best model
        output_dir = './model_save/'
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print('Early stopping triggered.')
        break

training_end_time = time.time()
training_time = training_end_time - training_start_time

print(f"Training complete! Total training time: {training_time:.2f} seconds")

# Load the best model
best_model_dir = './model_save/'

model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
model.to(device)

# Evaluate on the test set
print("Running Test Set Evaluation...")

model.eval()
predictions, true_labels = [], []

test_start_time = time.time()

for batch in tqdm(test_dataloader, desc="Testing", leave=False):
    b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)

    with torch.no_grad():
        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_input_mask
        )

    logits = outputs.logits
    preds = torch.argmax(logits, dim=1).flatten()

    predictions.extend(preds.cpu().numpy())
    true_labels.extend(b_labels.cpu().numpy())

test_end_time = time.time()
test_time = test_end_time - test_start_time

print(f"Test complete! Total prediction time: {test_time:.2f} seconds")

# Classification report
print(classification_report(true_labels, predictions, digits=4))

# Confusion matrix
cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Bar plot for metrics
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

metrics = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1 Score': f1}
plt.figure(figsize=(8, 6))
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()))
plt.ylim(0, 1)
plt.title('Metrics')
plt.show()

# Plot the learning curve
sns.set(style='darkgrid')
sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)
plt.plot(loss_values, 'b-o', label="Training loss")
plt.plot(validation_loss_values, 'r-o', label="Validation loss")
plt.title("Learning Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


Mounted at /content/drive
Device: cuda
GPU Name: Tesla T4


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Average training loss: 0.6944
Running Validation...
Validation Loss: 0.6936
Validation Accuracy: 49.82%
======== Epoch 3 / 10 ========
Training...


Training:   0%|          | 0/1875 [00:00<?, ?it/s]

Average training loss: 0.6941
Running Validation...
Validation Loss: 0.6938
Validation Accuracy: 50.18%
======== Epoch 4 / 10 ========
Training...


Training:   0%|          | 0/1875 [00:00<?, ?it/s]

Average training loss: 0.6940
Running Validation...
